# 04 — Orange held-out evaluation and example run

Evaluate the supplied test split once, after selecting models using training/validation data.

In [ ]:
from pathlib import Path
import sys, time, joblib, numpy as np, torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Open this notebook from the project folder or one of its subfolders.')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset import torch_datasets
from src.dl_model import load_model
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
bundle=joblib.load(PROJECT_ROOT / 'data' / 'processed' / 'orange_test_features.joblib')
X_test,y_test,class_names=bundle['features'],bundle['labels'],bundle['class_names']
classical=joblib.load(PROJECT_ROOT / 'models' / 'classical_best.joblib')['model']
pred_classical=classical.predict(X_test)
print('Classical test accuracy:',accuracy_score(y_test,pred_classical))
print(classification_report(y_test,pred_classical,target_names=class_names,zero_division=0))
print('Classical confusion matrix\n',confusion_matrix(y_test,pred_classical))

In [ ]:
device='cuda' if torch.cuda.is_available() else 'cpu'
model=load_model(PROJECT_ROOT / 'models' / 'mobilenetv2_best.pth',num_classes=len(class_names),device=device)
(_,_,test_ds), _=torch_datasets(DATA_ROOT)
test_loader=DataLoader(test_ds,batch_size=32,shuffle=False,num_workers=0)
truth=[]; predictions=[]; elapsed=[]
model.eval()
with torch.no_grad():
    for images,labels in test_loader:
        images=images.to(device); start=time.perf_counter(); logits=model(images); elapsed.append(time.perf_counter()-start)
        predictions.extend(logits.argmax(1).cpu().numpy()); truth.extend(labels.numpy())
print('MobileNetV2 test accuracy:',accuracy_score(truth,predictions))
print(classification_report(truth,predictions,target_names=class_names,zero_division=0))
print('MobileNetV2 confusion matrix\n',confusion_matrix(truth,predictions))
print('Mean batch inference seconds:',np.mean(elapsed))
print('Report macro-F1 and weighted-F1; macro-F1 is the primary comparison when class balance differs.')

In [ ]:
from src.dataset import discover_images
from main_demo import predict
example,_=discover_images(DATA_ROOT / 'test')[0]
label,score,scores=predict(example,model,device,class_names)
print('Image:',example)
print('Predicted class:',label,'score:',round(score,3))
print(scores)

## Required assignment questions

1. **Production performance:** report the measured held-out and preferably external-site results, class-wise recall, latency on target hardware, and drift monitoring. A random image split does not establish production performance.
2. **Drawbacks:** visually similar stages, lighting/background sensitivity, class imbalance, limited data, potential leakage, and lack of calibration/domain validation.
3. **With more time:** gather a larger multi-site dataset; group splits by fruit/session; compare repeated seeds; tune/fine-tune; calibrate confidence; test on new cameras and varieties; optimize deployment latency.
4. **Where time was spent:** record actual effort during the project. A typical emphasis is data validation/label review and leakage-safe evaluation, because those determine whether model scores are trustworthy.